In [ ]:
!nvidia-smi

In [ ]:
!pip install accelerate
!pip install "transformers==4.43.0" "huggingface_hub==0.34.1" bitsandbytes


In [ ]:
from huggingface_hub import list_repo_files
# finds possible 7b qwen 2.5 models
files = list_repo_files("Qwen/Qwen2.5-7B-Instruct-GGUF")
for f in files:
    if f.endswith(".gguf"):
        print(f)


In [ ]:
from huggingface_hub import hf_hub_download
# downloads 4 bit quantized qwen 2.5 7b model
repo_id = "Qwen/Qwen2.5-7B-Instruct-GGUF"

files = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]

paths = []
for f in files:
    paths.append(
        hf_hub_download(repo_id=repo_id, filename=f)
    )

paths


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16  # T4 needs float16, not bfloat16
)

inputs = tokenizer("Hello world", return_tensors="pt")
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# input
inputs = tokenizer("Hello world", return_tensors="pt")

# output in tokens
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode
print(tokenizer.decode(outputs[0]))
